# Sports14 Text/Image Feature Extraction

In [1]:

import os
import numpy as np
import pandas as pd

In [4]:
dataset = 'toys'
os.chdir(f'/home/raozhongtao/MMRec/data/{dataset}')
os.getcwd()
image_feat = np.load('./image_feat.npy')
print(image_feat.shape)

(68556, 4096)


## Load text data

In [21]:
i_id, desc_str = 'itemID', 'description'

file_path = './'
file_name = f'meta-{dataset}.csv'

meta_file = os.path.join(file_path, file_name)

df = pd.read_csv(meta_file)
df.sort_values(by=[i_id], inplace=True)

print('data loaded!')
print(f'shape: {df.shape}')

df[:3]

data loaded!
shape: (68556, 10)


,itemID,asin,description,title,price,salesRank,imUrl,brand,categories,related
0,0,B000E9DPCW,It's truly a FRESH START for puzzles! This han...,Melissa &amp; Doug Farm Wooden Chunky Puzzle,9.89,{'Toys & Games': 595},http://ecx.images-amazon.com/images/I/51HwF3vH...,Melissa &amp; Doug,"[['Toys & Games', 'Puzzles']]","{'also_bought': ['B000E9DPVI', 'B000F676D8', '..."
1,1,B000F676D8,It's truly a FRESH START for puzzles! This han...,Melissa &amp; Doug Shapes - Chunky Puzzle,11.19,{'Toys & Games': 1478},http://ecx.images-amazon.com/images/I/511V1HGN...,Melissa &amp; Doug,"[['Toys & Games', 'Puzzles', 'Pegged Puzzles']]","{'also_bought': ['B000E9DPCW', 'B000E9DPVI', '..."
2,2,0375829695,"A collection of six 48-piece (that is,slightly...",Dr. Seuss Jigsaw Puzzle Book: With Six 48-Piec...,24.82,{'Home &amp; Kitchen': 590975},http://ecx.images-amazon.com/images/I/51Q02ZH6...,Dr. Seuss,"[['Toys & Games', 'Puzzles', 'Jigsaw Puzzles']]","{'also_viewed': ['1865036013', 'B004UB2DV4', '..."


In [22]:

# sentences: title + brand + category + description | All have title + description

title_na_df = df[df['title'].isnull()]
print(title_na_df.shape)

desc_na_df = df[df['description'].isnull()]
print(desc_na_df.shape)

na_df = df[df['description'].isnull() & df['title'].isnull()]
print(na_df.shape)

na3_df = df[df['description'].isnull() & df['title'].isnull() & df['brand'].isnull()]
print(na3_df.shape)

na4_df = df[df['description'].isnull() & df['title'].isnull() & df['brand'].isnull() & df['categories'].isnull()]
print(na4_df.shape)

(220, 10)
(3550, 10)
(117, 10)
(117, 10)
(0, 10)


In [23]:

df[desc_str] = df[desc_str].fillna(" ")
df['title'] = df['title'].fillna(" ")
df['brand'] = df['brand'].fillna(" ")
df['categories'] = df['categories'].fillna(" ")


In [24]:
sentences = []
for i, row in df.iterrows():
    sen = row['title'] + ' ' + row['brand'] + ' '
    cates = eval(row['categories'])
    if isinstance(cates, list):
        for c in cates[0]:
            sen = sen + c + ' '
    sen += row[desc_str]
    sen = sen.replace('\n', ' ')

    sentences.append(sen)

sentences[:10]

["Melissa &amp; Doug Farm Wooden Chunky Puzzle Melissa &amp; Doug Toys & Games Puzzles It's truly a FRESH START for puzzles! This hand-painted, playfully styled puzzle is like nothing you've seen before!  The thick, chunky wooden puzzle pieces fit neatly into their spots on the colorful board and also stand up for additional pretend play.  Full-color pictures beneath each piece.",
 "Melissa &amp; Doug Shapes - Chunky Puzzle Melissa &amp; Doug Toys & Games Puzzles Pegged Puzzles It's truly a FRESH START for puzzles! This hand-painted; playfully styled puzzle is like nothing you've seen before!  The thick; chunky wooden puzzle pieces fit neatly into their spots on the colorful board and also stand up for additional pretend play.  Full-color pictures beneath each piece.",
 'Dr. Seuss Jigsaw Puzzle Book: With Six 48-Piece Puzzles Dr. Seuss Toys & Games Puzzles Jigsaw Puzzles A collection of six 48-piece (that is,slightlychallenging), sturdy, jigsaw puzzles   featuring artwork and text from

In [25]:

course_list = df[i_id].tolist()
#sentences = df[desc_str].tolist()

assert course_list[-1] == len(course_list) - 1

In [26]:
# should `pip install sentence_transformers` first
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

sentence_embeddings = model.encode(sentences)
print('text encoded!')

assert sentence_embeddings.shape[0] == df.shape[0]
np.save(os.path.join(file_path, 'text_feat.npy'), sentence_embeddings)
print('done!')


text encoded!
done!


In [27]:
sentence_embeddings[:10]

array([[-0.01178961, -0.02778944,  0.0661981 , ...,  0.08479404,
         0.06026963,  0.04514093],
       [-0.00785961, -0.03747069,  0.04839188, ...,  0.08953275,
         0.0809304 ,  0.05758647],
       [-0.03554062,  0.05227063,  0.06799965, ...,  0.05868177,
         0.04887627,  0.05242642],
       ...,
       [-0.04855203,  0.04069882, -0.07028265, ..., -0.0962716 ,
         0.06227377,  0.06183891],
       [-0.01801053, -0.02111114,  0.03638645, ..., -0.01652003,
         0.02553844, -0.0158672 ],
       [-0.06646657,  0.03749579,  0.04303032, ...,  0.02412607,
         0.0411323 ,  0.11716458]], dtype=float32)

In [28]:
load_txt_feat = np.load('text_feat.npy', allow_pickle=True)
print(load_txt_feat.shape)
load_txt_feat[:10]

(68556, 384)


array([[-0.01178961, -0.02778944,  0.0661981 , ...,  0.08479404,
         0.06026963,  0.04514093],
       [-0.00785961, -0.03747069,  0.04839188, ...,  0.08953275,
         0.0809304 ,  0.05758647],
       [-0.03554062,  0.05227063,  0.06799965, ...,  0.05868177,
         0.04887627,  0.05242642],
       ...,
       [-0.04855203,  0.04069882, -0.07028265, ..., -0.0962716 ,
         0.06227377,  0.06183891],
       [-0.01801053, -0.02111114,  0.03638645, ..., -0.01652003,
         0.02553844, -0.0158672 ],
       [-0.06646657,  0.03749579,  0.04303032, ...,  0.02412607,
         0.0411323 ,  0.11716458]], dtype=float32)

# Image encoder (V0)，following LATTICE, averaging over for missed items

In [29]:
df[:5]

,itemID,asin,description,title,price,salesRank,imUrl,brand,categories,related
0,0,B000E9DPCW,It's truly a FRESH START for puzzles! This han...,Melissa &amp; Doug Farm Wooden Chunky Puzzle,9.89,{'Toys & Games': 595},http://ecx.images-amazon.com/images/I/51HwF3vH...,Melissa &amp; Doug,"[['Toys & Games', 'Puzzles']]","{'also_bought': ['B000E9DPVI', 'B000F676D8', '..."
1,1,B000F676D8,It's truly a FRESH START for puzzles! This han...,Melissa &amp; Doug Shapes - Chunky Puzzle,11.19,{'Toys & Games': 1478},http://ecx.images-amazon.com/images/I/511V1HGN...,Melissa &amp; Doug,"[['Toys & Games', 'Puzzles', 'Pegged Puzzles']]","{'also_bought': ['B000E9DPCW', 'B000E9DPVI', '..."
2,2,0375829695,"A collection of six 48-piece (that is,slightly...",Dr. Seuss Jigsaw Puzzle Book: With Six 48-Piec...,24.82,{'Home &amp; Kitchen': 590975},http://ecx.images-amazon.com/images/I/51Q02ZH6...,Dr. Seuss,"[['Toys & Games', 'Puzzles', 'Jigsaw Puzzles']]","{'also_viewed': ['1865036013', 'B004UB2DV4', '..."
3,3,B000AS2AL4,The sort of learning fun that kids need! Matc...,Melissa &amp; Doug Stack and Sort Board,9.99,{'Toys & Games': 3406},http://ecx.images-amazon.com/images/I/31SS-pAf...,Melissa &amp; Doug,"[['Toys & Games', 'Baby & Toddler Toys', 'Stac...","{'also_bought': ['B000067PWG', 'B00462PTZ4', '..."
4,4,B000C26AC8,International Playthings iPlay Baby Activity P...,International Playthings iPlay Baby Activity P...,NaN,{'Toys & Games': 312210},http://ecx.images-amazon.com/images/I/51tC5yvZ...,,"[['Toys & Games', 'Sports & Outdoor Play', 'Po...","{'also_viewed': ['B000L5PKVI', 'B009QUGMTI', '..."


In [30]:
import array

def readImageFeatures(path):
    with open(path, 'rb') as f:
        while True:
            asin = f.read(10).decode('UTF-8')  # Read 10-byte ASIN
            if asin == '':
                break
            a = array.array('f')
            try:
                a.fromfile(f, 4096)  # Read 4096 floats
                yield asin, a.tolist()
            except EOFError:
                print(f"Warning: Incomplete data for ASIN {asin}, skipping.")
                break


In [ ]:

img_data = readImageFeatures(f"image_features_{dataset}.b")


item2id = dict(zip(df['asin'], df['itemID']))

feats = {}
avg = []
for d in img_data:
    if d[0] in item2id:
        feats[int(item2id[d[0]])] = d[1]
        avg.append(d[1])
avg = np.array(avg).mean(0).tolist()

ret = []
non_no = []
for i in range(len(item2id)):
    if i in feats:
        ret.append(feats[i])
    else:
        non_no.append(i)
        ret.append(avg)

print('# of items not in processed image features:', len(non_no))
assert len(ret) == len(item2id)
np.save('image_feat.npy', np.array(ret))
np.savetxt("missed_img_itemIDs.csv", non_no, delimiter =",", fmt ='%d')
print('done!')

# of items not in processed image features: 544
done!
